In [1]:
import warnings
warnings.filterwarnings('ignore')

# Imports

In [2]:
import torch
import scanpy as sc
import numpy as np
import anndata as ad
from tqdm import tqdm
import os

from scdisentangle.train.tools import get_trainer, set_seed

# Params

In [3]:
seed_nb = 42
yaml_path = '../configs/kang_disentangle.yaml'
weights_path =  '../weights/MIG_BINNED_dis_latent_stack_condition_train'

counterfactual_dict = {
    'condition': 'control'
}

covariate_name = 'cell_type'

# Set seed

In [4]:
set_seed(seed_nb)

ic| 'Setting seed to', seed: 42


# Get trainer

In [5]:
trainer = get_trainer(yaml_path, wandb_log=False)
trainer.load_weights(weights_path)

Global seed set to 0
ic| 'Setting seed to', seed: 42
ic| 'Creating cell mappings'
ic| 'Creating inputs'
ic| 'Creating inputs'


Wandb is off
Loading weights from ../weights/MIG_BINNED_dis_latent_stack_condition_train


# Predict to get latent

In [6]:
adata = trainer.predict(
    trainer.dataset.data.copy(), 
    counterfactual_dict={}, 
    bs=256
)
adata.layers['org_expression'] = trainer.dataset.data.X.copy()

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 54/54 [00:00<00:00, 87.92it/s]


# Get progressive latent (and optionally recs)

In [7]:
from scdisentangle.train.progressive_latent import get_progressive_latent

In [8]:
# Balanced
adata_structured_balanced = get_progressive_latent(
    trainer=trainer,
    adata=adata,
    counterfactual_dict=counterfactual_dict,
    get_recs=True,
    balance_clusters=True,
    covariate_name='cell_type',
    )

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:02<00:00,  7.11it/s]


In [9]:
# Unbalanced
adata_structured_unbalanced = get_progressive_latent(
    trainer=trainer,
    adata=adata,
    counterfactual_dict=counterfactual_dict,
    get_recs=True,
    balance_clusters=False,
    covariate_name='cell_type',
    )

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 16/16 [00:07<00:00,  2.06it/s]


# Save adata with latent levels

In [10]:
adata_structured_balanced.write_h5ad('adata_structured_balanced.h5ad')
adata_structured_unbalanced.write_h5ad('adata_structured_unbalanced.h5ad')